In [10]:
# extract
import numpy as np
from numpy import linalg as LA

from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input

class VGGNet:
    def __init__(self):
        self.input_shape = (224, 224, 3)
        self.weight = 'imagenet'
        self.pooling = 'max'
        self.model = VGG16(weights = self.weight, input_shape = (self.input_shape[0], self.input_shape[1], self.input_shape[2]), pooling = self.pooling, include_top = False)
        self.model.predict(np.zeros((1, 224, 224 , 3)))

    def extract_feat(self, img_path):
        img = image.load_img(img_path, target_size=(self.input_shape[0], self.input_shape[1]))
        img = image.img_to_array(img)
        img = np.expand_dims(img, axis=0)
        img = preprocess_input(img)
        feat = self.model.predict(img)
        norm_feat = feat[0]/LA.norm(feat[0])
        return norm_feat


In [11]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [58]:
import os
import h5py
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [59]:
images_path ="/content/gdrive/MyDrive/ml/train/"

model = VGGNet()
path = "/content/gdrive/MyDrive/ml/train/"

feats = []
names = []

# for im in os.listdir(path):
#     X = model.extract_feat(path+im)
#     feats.append(X)
#     names.append(im)
for root, dirs, files in os.walk(images_path):
    for f in files:
      full_path = os.path.join(root, f)
      X = model.extract_feat(full_path)
      feats.append(X)
      rel_path = os.path.relpath(full_path, images_path)
      names.append(rel_path)
feats = np.array(feats)
output = "VGG16Features.h5"

h5f = h5py.File(output, 'w')
h5f.create_dataset('dataset_1', data = feats)
h5f.create_dataset('dataset_2', data = np.bytes_(names))
h5f.close()

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 713ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 512ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 696ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 511ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 497ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 503ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 501ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 481ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 508ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 540ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 849ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 865ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 860ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 739ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 492ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 490ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 517ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 518ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 490ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 488ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 517ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 491ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 531ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 492ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 513ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 

In [60]:
import h5py

# เปิดไฟล์ HDF5
with h5py.File('VGG16Features.h5', 'r') as h5f:
    # ตรวจสอบ keys ในไฟล์
    print("Keys in the HDF5 file:", list(h5f.keys()))

    # ตรวจสอบข้อมูลใน dataset_2 ก่อน
    dataset_2 = h5f['dataset_2']
    print(f"Dataset_2 info: {dataset_2}")

    # ถ้า dataset_2 เป็น scalar หรือเป็น array ที่ไม่สามารถ slice ได้
    if dataset_2.shape == ():
        # ถ้าเป็น scalar, จะดึงค่ามาเป็นค่าคงที่
        print("Dataset_2 is scalar, can't slice it.")
        names = dataset_2[()]
    else:
        # ถ้าเป็น array, สามารถ slice ได้
        names = dataset_2[:]

    print(f"Names of images (dataset_2): {names}")

    # ตัวอย่างข้อมูลแรกในแต่ละ dataset
    feats = h5f['dataset_1'][:]
    print(f"First feature vector: {feats[0]}")


Keys in the HDF5 file: ['dataset_1', 'dataset_2']
Dataset_2 info: <HDF5 dataset "dataset_2": shape (2872,), type "|S191">
Names of images (dataset_2): [b'Roman Colosseum - Rome/101.jpg' b'Roman Colosseum - Rome/103.jpg'
 b'Roman Colosseum - Rome/102.jpg' ...
 b'Himalaya - India/25.discover-himalaya-banner-2.jpg'
 b'Himalaya - India/28.Chhota-Shigri-Glacier-5300-m-a.s.l-Copy.jpg'
 b'Himalaya - India/24.mount-kamet.jpg']
First feature vector: [5.41783907e-02 6.98100822e-03 1.80752203e-02 0.00000000e+00
 8.12515095e-02 0.00000000e+00 5.16163856e-02 0.00000000e+00
 0.00000000e+00 6.15301766e-02 0.00000000e+00 0.00000000e+00
 2.33774949e-02 0.00000000e+00 2.36378051e-02 0.00000000e+00
 0.00000000e+00 2.84511577e-02 0.00000000e+00 7.68147185e-02
 5.20643778e-02 3.79661727e-03 8.97256657e-03 0.00000000e+00
 0.00000000e+00 4.11822759e-02 2.81738373e-03 0.00000000e+00
 0.00000000e+00 0.00000000e+00 4.20948565e-02 0.00000000e+00
 2.78333928e-02 0.00000000e+00 7.50337169e-02 1.82928313e-02
 0.000

In [61]:
with h5py.File('VGG16Features.h5', 'r') as h5f:
    # ตรวจสอบ keys ในไฟล์
    print("Keys in the HDF5 file:", list(h5f.keys()))

    # โหลด dataset_1 (ฟีเจอร์)
    feats = h5f['dataset_1'][:]
    print(f"Shape of features dataset (dataset_1): {feats.shape}")

Keys in the HDF5 file: ['dataset_1', 'dataset_2']
Shape of features dataset (dataset_1): (2872, 512)


In [62]:
 with h5py.File('VGG16Features.h5', 'r') as h5f:
      # โหลด dataset_2 (ชื่อภาพ)
    names = h5f['dataset_2'][:]
    print(f"Names of images (dataset_2): {names}")

Names of images (dataset_2): [b'Roman Colosseum - Rome/101.jpg' b'Roman Colosseum - Rome/103.jpg'
 b'Roman Colosseum - Rome/102.jpg' ...
 b'Himalaya - India/25.discover-himalaya-banner-2.jpg'
 b'Himalaya - India/28.Chhota-Shigri-Glacier-5300-m-a.s.l-Copy.jpg'
 b'Himalaya - India/24.mount-kamet.jpg']


In [63]:
 with h5py.File('VGG16Features.h5', 'r') as h5f:
    # ตัวอย่างข้อมูลแรกในแต่ละ dataset
    print(f"First feature vector: {feats[0]}")
    print(f"First image name: {names[0]}")

First feature vector: [5.41783907e-02 6.98100822e-03 1.80752203e-02 0.00000000e+00
 8.12515095e-02 0.00000000e+00 5.16163856e-02 0.00000000e+00
 0.00000000e+00 6.15301766e-02 0.00000000e+00 0.00000000e+00
 2.33774949e-02 0.00000000e+00 2.36378051e-02 0.00000000e+00
 0.00000000e+00 2.84511577e-02 0.00000000e+00 7.68147185e-02
 5.20643778e-02 3.79661727e-03 8.97256657e-03 0.00000000e+00
 0.00000000e+00 4.11822759e-02 2.81738373e-03 0.00000000e+00
 0.00000000e+00 0.00000000e+00 4.20948565e-02 0.00000000e+00
 2.78333928e-02 0.00000000e+00 7.50337169e-02 1.82928313e-02
 0.00000000e+00 1.11918859e-02 0.00000000e+00 0.00000000e+00
 1.22857857e-02 9.25454311e-03 4.56428267e-02 2.22587809e-02
 3.54136638e-02 9.39118341e-02 0.00000000e+00 2.34707762e-02
 1.99218933e-02 2.06664158e-03 7.18411207e-02 2.66193729e-02
 1.15424721e-02 2.41100136e-02 8.07481036e-02 4.43660840e-02
 1.99265536e-02 0.00000000e+00 1.14647903e-01 1.85253676e-02
 6.45253155e-03 0.00000000e+00 8.19056761e-03 5.80555238e-02
 0